<a href="https://colab.research.google.com/github/svasan/public_colab/blob/main/Ni%C3%B1o_3_4_SST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl -L -A "Mozilla/5.0" \
  -e "https://climatereanalyzer.org/clim/sst_daily/?dm_id=nino3.4" \
  -o oisst2.1_nino3.4_sst_day.json \
  "https://climatereanalyzer.org/clim/sst_daily/json_2clim/oisst2.1_nino3.4_sst_day.json"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  123k  100  123k    0     0   251k      0 --:--:-- --:--:-- --:--:--  251k


In [ ]:
# Cell 2: parse data and calculate standard deviations from the 1991-2020 daily mean

import json
import numpy as np
import pandas as pd

with open("oisst2.1_nino3.4_sst_day.json", "r") as f:
    raw = json.load(f)

records = []
clim_1991_2020 = None
prelim = None

for series in raw:
    name = str(series["name"])
    values = series["data"]

    if name == "1991-2020":
        clim_1991_2020 = pd.Series(values, index=np.arange(1, len(values) + 1), dtype="float64")
    elif name.lower() == "preliminary":
        prelim = pd.Series(values, index=np.arange(1, len(values) + 1), dtype="float64")
    elif name.isdigit():
        year = int(name)
        for day, value in enumerate(values, start=1):
            records.append({"year": year, "day": day, "sst_c": value})

df = pd.DataFrame(records)
df["sst_c"] = pd.to_numeric(df["sst_c"], errors="coerce")

# Merge preliminary values into the most recent year where available.
# Climate Reanalyzer publishes the last ~2 weeks separately as "Preliminary".
if prelim is not None:
    latest_year = df.loc[df["sst_c"].notna(), "year"].max()
    prelim_df = pd.DataFrame({
        "year": latest_year,
        "day": prelim.index,
        "sst_c_prelim": prelim.values
    })
    df = df.merge(prelim_df, on=["year", "day"], how="left")
    df["sst_c"] = df["sst_c_prelim"].combine_first(df["sst_c"])
    df = df.drop(columns=["sst_c_prelim"])

baseline = df[df["year"].between(1991, 2020)].copy()

daily_sd = (
    baseline
    .groupby("day")["sst_c"]
    .std(ddof=1)
    .replace(0, np.nan)
)

clim = clim_1991_2020.rename("clim_mean_c").to_frame()
clim["clim_sd_c"] = daily_sd

plot_df = df.merge(clim, left_on="day", right_index=True, how="left")
plot_df["std_dev"] = (plot_df["sst_c"] - plot_df["clim_mean_c"]) / plot_df["clim_sd_c"]

# Optional: drop day 366 if you want every year on the same 365-day axis.
# The original-like chart usually looks cleaner this way.
plot_df = plot_df[plot_df["day"] <= 365].copy()

current_year = int(plot_df.loc[plot_df["std_dev"].notna(), "year"].max())
current_year

2026

In [ ]:
# Cell 3: interactive Plotly reproduction
# Hover over any blue line to see the year, day of year, SST, and standard deviation.

import plotly.graph_objects as go

fig = go.Figure()

years = sorted(plot_df["year"].dropna().unique())

for year in years:
    year = int(year)
    ydf = plot_df[plot_df["year"] == year].sort_values("day")
    is_current = year == current_year

    fig.add_trace(
        go.Scatter(
            x=ydf["day"],
            y=ydf["std_dev"],
            mode="lines",
            name=str(year),
            line=dict(
                color="#c5161d" if is_current else "rgba(31, 91, 125, 0.42)",
                width=3 if is_current else 1,
            ),
            opacity=1.0 if is_current else 0.75,
            hovertemplate=(
                "Year: %{customdata[0]}<br>"
                "Day of year: %{x}<br>"
                "SST: %{customdata[1]:.3f} °C<br>"
                "Std dev from 1991-2020 mean: %{y:.2f}<extra></extra>"
            ),
            customdata=np.column_stack([ydf["year"], ydf["sst_c"]]),
            showlegend=bool(is_current),
        )
    )

fig.add_hline(y=0, line_width=1, line_color="black")

fig.update_layout(
    title=(
        f"Niño 3.4 Sea-Surface Temperatures: 1982-{current_year}<br>"
        "<sup>Standard deviations from 1991-2020 daily mean</sup>"
    ),
    xaxis_title="Day of Year",
    yaxis_title="Standard Deviation",
    template="plotly_white",
    width=1050,
    height=720,
    hovermode="closest",
    margin=dict(l=70, r=30, t=90, b=70),
)

fig.update_xaxes(range=[1, 365], showgrid=False)
fig.update_yaxes(range=[-4, 4], dtick=0.5, zeroline=False)

fig.add_annotation(
    text="Data: https://climatereanalyzer.org/clim/sst_daily/json_2clim/oisst2.1_nino3.4_sst_day.json",
    xref="paper",
    yref="paper",
    x=0.5,
    y=0.01,
    showarrow=False,
    font=dict(size=10, color="black"),
)

fig.show()